In [1]:
import rclpy
from rclpy.node import Node

import math
import random

from PIL import Image, ImageOps
import numpy as np
from geometry_msgs.msg import PointStamped, Twist
from nav_msgs.msg import Odometry
from visualization_msgs.msg import Marker


import numpy as np
#from tf_transformations import euler_from_quaternion, quaternion_inverse, quaternion_multiply

def euler_from_quaternion(quaternion):
    """
    Converts quaternion (w in last place) to euler roll, pitch, yaw
    quaternion = [x, y, z, w]
    Bellow should be replaced when porting for ROS 2 Python tf_conversions is done.
    """
    x = quaternion.x
    y = quaternion.y
    z = quaternion.z
    w = quaternion.w

    sinr_cosp = 2 * (w * x + y * z)
    cosr_cosp = 1 - 2 * (x * x + y * y)
    roll = np.arctan2(sinr_cosp, cosr_cosp)

    sinp = 2 * (w * y - z * x)
    pitch = np.arcsin(sinp)

    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    yaw = np.arctan2(siny_cosp, cosy_cosp)

    return roll, pitch, yaw

class MarkerPublisher(Node):
    
    def __init__(self):
        super().__init__('marker_handler_node')
        self.subscription_click = self.create_subscription(
            PointStamped,
            '/clicked_point',
            self.listener_callback,
            10   
        )

        self.subscription_pos = self.create_subscription(
            Odometry,
            '/model/jetbot/odometry',
            self.odom_callback,
            10
        )

        self.x = 1190
        self.y = 80
        self.yaw = 0.0
        img = Image.open('/ros2_ws/map.png')
        img = ImageOps.grayscale(img)
        img = ImageOps.invert(img)
        # Inicjalizacja mapy dla RRT (musisz podać swoją macierz/obraz mapy)
        self.mapa = np.array(img)
        
        # Lista przechowująca krotki (x, y) wyznaczonej ścieżki
        self.path_points = []
        # Aktualny podcel, do którego zmierza robot
        self.current_subgoal = None

        self.publisher_click = self.create_publisher(Marker, '/visualization_marker', 10)
        self.publisher_odom = self.create_publisher(Marker, '/odom_marker', 10)
        self.publisher_cmd = self.create_publisher(Twist, '/cmd_vel', 10)
        goal_pos = (100,800)
        self.get_logger().info(f"Obliczanie ścieżki RRT z {(self.x,self.y)} do {goal_pos}...")
        raw_path = rrt_solver((self.x,self.y), goal_pos, self.mapa, is_rrt_star=True)

        if raw_path is not None:
            self.get_logger().info("Znaleziono ścieżkę RRT!")
            # Odwracamy listę, ponieważ oryginalna była od celu do startu
            self.path_points = raw_path[::-1]
            
            # Opcjonalnie: usuwamy pierwszy punkt, bo to obecna pozycja robota
            if len(self.path_points) > 0:
                self.path_points.pop(0)
                
            self.get_logger().info(f"Liczba punktów w ścieżce: {len(self.path_points)}")
        else:
            self.get_logger().warn("RRT: Nie udało się znaleźć ścieżki!")
            self.path_points = []
            self.current_subgoal = None
    

    def listener_callback(self, msg):
        marker = Marker()
        marker.header.frame_id = "map"
        marker.header.stamp = self.get_clock().now().to_msg()
        marker.type = Marker.SPHERE
        marker.action = Marker.ADD

        marker.ns = 'clicker'
        marker.id = 0
        marker.pose.position = msg.point

        marker.scale.x = 0.2
        marker.scale.y = 0.2
        marker.scale.z = 0.2
        marker.color.a = 1.0
        marker.color.r = 1.0

        self.publisher_click.publish(marker)

        # Definiujemy start (pozycja robota) i cel (kliknięty punkt)
        start_pos = (self.x, self.y)
        goal_pos = (msg.point.x, msg.point.y)

        self.get_logger().info(f"Obliczanie ścieżki RRT z {start_pos} do {goal_pos}...")
        
        # Wywołanie Twojej funkcji RRT
        # Przekaż obiekt 'self.mapa' lub globalną zmienną mapy
        raw_path = rrt_solver(start_pos, goal_pos, self.mapa, is_rrt_star=True)

        if raw_path is not None:
            self.get_logger().info("Znaleziono ścieżkę RRT!")
            # Odwracamy listę, ponieważ oryginalna była od celu do startu
            self.path_points = raw_path[::-1]
            
            # Opcjonalnie: usuwamy pierwszy punkt, bo to obecna pozycja robota
            if len(self.path_points) > 0:
                self.path_points.pop(0)
                
            self.get_logger().info(f"Liczba punktów w ścieżce: {len(self.path_points)}")
        else:
            self.get_logger().warn("RRT: Nie udało się znaleźć ścieżki!")
            self.path_points = []
            self.current_subgoal = None


    def odom_callback(self, msg):
        marker = Marker()
        marker.header.frame_id = "map"
        marker.header.stamp = self.get_clock().now().to_msg()
        marker.type = Marker.CYLINDER
        marker.action = Marker.ADD

        marker.ns = 'odom'
        marker.id = 1

        self.x = msg.pose.pose.position.x
        self.y = msg.pose.pose.position.y

        self.yaw = euler_from_quaternion(msg.pose.pose.orientation)[2]

        marker.pose.position.x = msg.pose.pose.position.x
        marker.pose.position.y = msg.pose.pose.position.y

        marker.scale.x = 0.2
        marker.scale.y = 0.2
        marker.scale.z = 0.2
        marker.color.a = 1.0
        marker.color.g = 1.0

        self.publisher_odom.publish(marker)

        # Uruchom pętlę sterowania, jeśli mamy punkty w kolejce lub realizujemy podcel
        if self.path_points or self.current_subgoal:
            self.control_loop()
    

    def control_loop(self):
        msg = Twist()

        # Jeśli nie mamy aktualnego podcelu, pobieramy następny z listy
        if self.current_subgoal is None and len(self.path_points) > 0:
            self.current_subgoal = self.path_points.pop(0)
            self.get_logger().info(f"Zmiana podcelu na: {self.current_subgoal}. Pozostało punktów: {len(self.path_points)}")

        # Jeśli lista jest pusta i nie ma podcelu, zatrzymaj robota
        if self.current_subgoal is None:
            msg.linear.x = 0.0
            msg.angular.z = 0.0
            self.publisher_cmd.publish(msg)
            return

        # Obliczanie wektora do obecnego podcelu (krotka: x, y)
        subgoal_x, subgoal_y = self.current_subgoal
        dx = subgoal_x - self.x
        dy = subgoal_y - self.y

        distance = np.sqrt(dx**2 + dy**2)
        angle_to_goal = np.arctan2(dy, dx)
        angle_error = angle_to_goal - self.yaw

        # Normalizacja błędu kąta do przedziału [-pi, pi]
        angle_error = np.arctan2(np.sin(angle_error), np.cos(angle_error))

        # Warunek osiągnięcia obecnego punktu (tolerancja np. 0.15 metra)
        if distance > 0.15:
            if abs(angle_error) > 0.2:
                msg.linear.x = 0.0
                msg.angular.z = 0.5 if angle_error > 0 else -0.5
            else:
                msg.linear.x = 0.3
                msg.angular.z = 1.5 * angle_error
        else:
            # Punkt osiągnięty, resetujemy sub-goal, aby w kolejnym kroku pobrać następny
            self.get_logger().info(f"Osiągnięto punkt: {self.current_subgoal}")
            self.current_subgoal = None
            msg.linear.x = 0.0
            msg.angular.z = 0.0

        self.publisher_cmd.publish(msg)

class treeNode():
    def __init__(self, locationX, locationY):
        self.locationX = locationX
        self.locationY = locationY
        self.children = []
        self.parent = None
        self.cost = 0.0

class Tree():
    def __init__(self, start_node):
        self.nodes = [start_node]


    def add_node(self, new_node, parent_node):
        self.nodes.append(new_node)
        parent_node.children.append(new_node)
        new_node.parent = parent_node
        return

    def nearest(self,new_node):
        best = None
        min_dist = float('inf')
        for node in self.nodes:
            dist = math.dist((node.locationX, node.locationY),(new_node.locationX, new_node.locationY))
            if dist < min_dist:
                min_dist = dist
                best = node

        return best

    def path_recovery(self,final_node):
        path = []

        while final_node.parent is not None:
            path.append(final_node)
            final_node = final_node.parent
        #path.reverse()
        return path

    def distance(self,node1,node2):
        return math.dist((node1.locationX,node1.locationY), (node2.locationX,node2.locationY))
    
    def near(self, node_mid, radius):
        nodes_list= []

        for node in self.nodes:
            if self.distance(node_mid, node) <= radius:
                nodes_list.append(node)

        return nodes_list
    
    def rewire(self, child, new_parent):

        if child.parent:
            child.parent.children.remove(child)
        child.parent = new_parent
        new_parent.children.append(child)
        
        return
    
    def update_descendant_costs(self, node):
        if node.parent:
            new_cost = node.parent.cost + self.distance(node.parent, node)
            
            if abs(node.cost - new_cost) > 1e-6:
                node.cost = new_cost
                
                for child in node.children:
                    self.update_descendant_costs(child)
        
        return


def collision_free(p1,p2,mapa):
    x1, y1 = p1.locationX, p1.locationY
    x2, y2 = p2.locationX, p2.locationY

    h, w = mapa.shape
    if not (0 <= x1 < w and 0 <= y1 < h and 0 <= x2 < w and 0 <= y2 < h):
        return False

    num_points = int(max(abs(x1-x2), abs(y1-y2)))

    if num_points == 0:
        return True

    for i in range(num_points + 1):
        t = i / max(num_points, 1)
        x = round(x1 + (x2 - x1) * t)
        y = round(y1 + (y2 - y1) * t)
        if mapa[y, x]  == 1:
            return False
    return True

def get_ellipse_params(start, goal, par_1=1.5, par_2=0.4):
    d = math.dist(start, goal)
    a = par_1 * d
    b = par_2 * d
    sr_x = (start[0] + goal[0]) / 2
    sr_y = (start[1] + goal[1]) / 2
    theta = math.atan2(goal[1] - start[1], goal[0] - start[0])
    return sr_x, sr_y, a, b, theta

def samp_point_elipse(start, goal, mapa, par_1 = 1.5, par_2 = 0.4):

    while True:
        d = math.dist(start,goal)
        a = par_1 * d
        b = par_2 * d
        sr_x = (start[0] + goal[0])  / 2
        sr_y = (start[1] + goal[1])  / 2
        theta = math.atan2(goal[1] - start[1], goal[0] - start[0])

        phi = random.uniform(0, 2 * math.pi)
        r = math.sqrt(random.uniform(0, 1))
        x_norm= a * r * math.cos(phi)
        y_norm = b * r * math.sin(phi)

        X = x_norm * math.cos(theta) - y_norm * math.sin(theta) + sr_x
        Y = x_norm * math.sin(theta) + y_norm * math.cos(theta) + sr_y

        if 0 <= X < mapa.shape[1] and 0 <= Y < mapa.shape[0]:
            if mapa[int(Y), int(X)] == 0:
                return treeNode(X, Y)

def rrt_solver(start, goal, mapa, step_len = 25, max_iter=1000, tolerance=30, bias_mode='FIXED', is_rrt_star=False, fixed_bias=0.9, pmax=0.4, pmin=0.05):
    h, w = mapa.shape

    start_node = treeNode(start[0], start[1])
    tree = Tree(start_node)

    sr_x, sr_y, a, b, theta = get_ellipse_params(start, goal)
    
    RRT_STAR_RADIUS = 5 * step_len
    success = 0
    pa = pmin

    for i in range(1, max_iter + 1):
        if bias_mode != 'FIXED':
            ps = success / i
            if ps < 1e-6:
                ps = 1e-6

            if bias_mode == 'AIW':
                pa = pmin + (pmax - pmin) * ps
            elif bias_mode == 'HYBRID':
                time_factor = (max_iter - i) / max_iter 
                term_1 = (pmax - pmin) / ps
                term_2 = pmin / ps
                pa = 1 - (term_1 + time_factor * term_2)
            pa = np.clip(pa, pmin, pmax)
        else: 
            pa = fixed_bias
            ps = success / i

        if random.random() < pa:
            sample_node = treeNode(goal[0], goal[1])
        else:
            sample_node = samp_point_elipse(start, goal, mapa)

        pot_parent = tree.nearest(sample_node)
        distance = tree.distance(pot_parent, sample_node)

        if distance <= step_len:
            new_node = sample_node
        else:
            stepX = ((sample_node.locationX - pot_parent.locationX) / distance) * step_len
            stepY = ((sample_node.locationY - pot_parent.locationY) / distance) * step_len
            new_node = treeNode(pot_parent.locationX + stepX, pot_parent.locationY + stepY)

        final_parent = pot_parent
        new_cost = 0.0
        if pot_parent:
            new_cost = pot_parent.cost + tree.distance(pot_parent, new_node)

        if is_rrt_star:
            near_nodes = tree.near(new_node, RRT_STAR_RADIUS)
            min_cost = new_cost
            best_parent = final_parent

            for near_node in near_nodes:
                potential_cost = near_node.cost + tree.distance(near_node, new_node)
                if potential_cost < min_cost and collision_free(near_node, new_node, mapa):
                    min_cost = potential_cost
                    best_parent = near_node
            
            final_parent = best_parent
            new_cost = min_cost
        
        new_node.cost = new_cost

        if final_parent and collision_free(final_parent, new_node, mapa):
            tree.add_node(new_node, final_parent)
            success += 1

            if is_rrt_star:
                for near_node in near_nodes:
                    rewire_cost = new_node.cost + tree.distance(new_node, near_node)
                    if rewire_cost < near_node.cost and collision_free(new_node, near_node, mapa):
                        tree.rewire(near_node, new_node)
                        tree.update_descendant_costs(near_node)

            # Sprawdzenie warunku końca (dotarcie do celu)
            goal_node = treeNode(goal[0], goal[1])
            if tree.distance(new_node, goal_node) < tolerance:
                if collision_free(new_node, goal_node, mapa):
                    tree.add_node(goal_node, new_node)
                    
                    # Pobieramy listę obiektów node z drzewa
                    found_path = tree.path_recovery(tree.nodes[-1])
                    
                    # Mapujemy obiekty node na listę krotek (x, y)
                    path_coordinates = [(node.locationX, node.locationY) for node in found_path]
                    return path_coordinates

    return None




def main(args=None):
    rclpy.init(args=args)
    node = MarkerPublisher()
    rclpy.spin(node)

    # Destroy the node explicitly
    # (optional - otherwise it will be done automatically
    # when the garbage collector destroys the node object)
    node.destroy_node()
    rclpy.shutdown()


if __name__ == '__main__':
    main()

ModuleNotFoundError: No module named 'rclpy'